<a href="https://colab.research.google.com/github/vifirsanova/ML-2026-pt-2/blob/main/advanced/1_recommender_systems/recsys_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Практическое домашнее задание. Группа Advanced (до 18 апреля, 23.59)

### Цель
Реализовать систему рекомендаций с использованием кластеризации на выбранном датасете и проверить качество через автоматические тесты.

### Задание

**Часть 1: Выбор и подготовка данных**
1. Выберите любой релевантный датасет:
   - MovieLens (фильмы)
   - Amazon Reviews (товары)
   - Last.fm (музыка)
   - Online Retail (товары)
   - Любой другой с числовыми признаками для кластеризации
2. Выберите минимум **4 числовых признака** для кластеризации
3. Проведите предобработку: очистка от пропусков, масштабирование

**Часть 2: Кластеризация**
1. Обучите модель KMeans с подбором количества кластеров
2. Обоснуйте выбор количества кластеров (графики Elbow и Silhouette)
3. Вычислите метрики качества: Silhouette Score, Davies-Bouldin Index

**Часть 3: Функция рекомендаций**
1. Реализуйте функцию `recommend(item_id, n_recommendations)`, которая возвращает топ-N похожих объектов из того же кластера
2. Функция должна исключать исходный объект из рекомендаций

**Часть 4: Тестирование**
Напишите тесты для проверки:
1. **Исполняемость кода** — все функции запускаются без ошибок
2. **Метрики качества** — Silhouette Score > 0.2, Davies-Bouldin < 1.5
3. **Корректность рекомендаций** — рекомендуемые объекты принадлежат тому же кластеру, исходный объект исключен

### Критерии оценки

| Критерий | Вес |
|----------|-----|
| Код исполняется без ошибок | 20% |
| Обоснованный выбор количества кластеров (графики + анализ) | 20% |
| Silhouette Score > 0.2 | 20% |
| Davies-Bouldin Index < 1.5 | 20% |
| Функция рекомендаций корректна + тесты проходят | 20% |

### Формат сдачи
- Jupyter Notebook с комментариями
- Отдельный файл с тестами (`test_clustering.py`)
- README с описанием датасета и результатами

## Тесты для проверки кода

> Тесты даются для референса и могут быть модифицрованы, особенно если вы используете ООП. Тем не менее рекомендуется использовать предоставленный код, либо сохранять предлагаемую структуру тестирования

In [ ]:
import pytest
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, davies_bouldin_score
import joblib
import os

# Тест 1: Проверка загрузки данных
def test_data_loading():
    df = pd.read_csv('student_dataset.csv')
    assert df is not None
    assert df.shape[0] > 100, "Датасет слишком маленький"
    assert not df.isnull().any().any(), "Есть пропуски в данных"

# Тест 2: Проверка масштабирования
def test_scaling():
    df = pd.read_csv('student_dataset.csv')
    feature_cols = ['danceability', 'energy', 'valence', 'tempo']
    X = df[feature_cols].dropna()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    assert np.allclose(X_scaled.mean(axis=0), 0, atol=1e-10)
    assert np.allclose(X_scaled.std(axis=0), 1, atol=1e-10)

# Тест 3: Проверка метрик качества
def test_clustering_quality():
    df = pd.read_csv('student_dataset.csv')
    feature_cols = ['danceability', 'energy', 'valence', 'tempo']
    X = df[feature_cols].dropna()

    kmeans = KMeans(n_clusters=8, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)

    silhouette = silhouette_score(X, labels)
    db_score = davies_bouldin_score(X, labels)

    assert silhouette > 0.2, f"Silhouette score {silhouette:.3f} < 0.2"
    assert db_score < 1.5, f"Davies-Bouldin score {db_score:.3f} > 1.5"

# Тест 4: Проверка функции рекомендаций
def test_recommendation_function():
    df = pd.read_csv('student_dataset.csv')
    df['cluster'] = pd.read_csv('clusters.csv')['cluster']

    def recommend(item_id, df, n=5):
        cluster = df.loc[item_id, 'cluster']
        candidates = df[df['cluster'] == cluster]
        return candidates.drop(item_id).head(n)

    recs = recommend(0, df)
    assert len(recs) == 5
    assert recs.index[0] != 0
    assert (recs['cluster'] == df.loc[0, 'cluster']).all()